In [ ]:
# ============================================================
# FINAL COMPARISON: 1D baselines vs 2D baselines vs temporal CNN
#
# Paste into one notebook cell after all deterministic models have finished.
# This cell DOES NOT retrain any model. It reads existing OOF predictions.
#
# Baseline result location requested by the user:
#   /mnt/home/qinqiu/thesis/result_baseline
# Main CNN result location:
#   /mnt/home/qinqiu/thesis/result_nested
#
# Outputs:
#   - Overall model comparison (patient + window level)
#   - Native-target and common-CNN-truth patient metrics
#   - Outer-fold metrics
#   - Pairwise patient-level comparisons among all 5 models
#   - Pairwise window-level comparisons only for compatible window definitions
#   - Wide prediction tables for later analysis
#   - Optional GA-group comparison if GA analysis files already exist
#   - Publication-style comparison figures
# ============================================================

from __future__ import annotations

import itertools
import json
import os
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

try:
    from scipy.stats import pearsonr, spearmanr, wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    pearsonr = spearmanr = wilcoxon = None
    warnings.warn("scipy is unavailable; correlation/test p-values will be NaN.")


# ============================================================
# 0. Configuration
# ============================================================
PROJECT_ROOT = "/mnt/home/qinqiu/thesis"
BASELINE_ROOT = os.path.join(PROJECT_ROOT, "result_baseline")
NESTED_ROOT = os.path.join(PROJECT_ROOT, "result_nested")

# Set an exact path here only when automatic discovery is ambiguous.
BASELINE_1D_DIR_OVERRIDE: Optional[str] = None
BASELINE_2D_DIR_OVERRIDE: Optional[str] = None
CNN_DIR_OVERRIDE: Optional[str] = None

EXPECTED_1D_BASENAME = (
    "baseline_1d_10min_ridge_xgboost_nestedcv_optuna_v2_mean1weights"
)
EXPECTED_2D_BASENAME = (
    "baseline_2d_flattened_ridge_xgboost_nestedcv_optuna_v2_mean1weights"
)
EXPECTED_CNN_BASENAME = (
    "nested_cv_120min_stride40min_v7_dataset_reuse_patientbalanced_resume100"
)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "result_comparison", "deterministic_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

BOOTSTRAP_REPEATS = 5000
BOOTSTRAP_SEED = 42
TRUE_TOLERANCE = 1e-5

MODEL_ORDER = [
    "Ridge 1D",
    "XGBoost 1D",
    "Ridge 2D",
    "XGBoost 2D",
    "CNN 120min",
]

MODEL_META = {
    "Ridge 1D": {
        "model_family": "Ridge",
        "input": "10-min 1D features + missing indicators + static",
        "window_definition": "10-min row-wise",
        "window_comparability_group": "1D_10min",
    },
    "XGBoost 1D": {
        "model_family": "XGBoost",
        "input": "10-min 1D features + missing indicators + static",
        "window_definition": "10-min row-wise",
        "window_comparability_group": "1D_10min",
    },
    "Ridge 2D": {
        "model_family": "Ridge",
        "input": "120-min flattened value + mask + static",
        "window_definition": "120-min, stride 40 min",
        "window_comparability_group": "2D_120min",
    },
    "XGBoost 2D": {
        "model_family": "XGBoost",
        "input": "120-min flattened value + mask + static",
        "window_definition": "120-min, stride 40 min",
        "window_comparability_group": "2D_120min",
    },
    "CNN 120min": {
        "model_family": "CNN",
        "input": "120-min temporal value + mask + static",
        "window_definition": "120-min, stride 40 min",
        "window_comparability_group": "2D_120min",
    },
}


# ============================================================
# 1. Path discovery
# ============================================================
def _required_prediction_files(directory: str, experiment_type: str) -> bool:
    if experiment_type in {"1d", "2d"}:
        required = [
            "FINAL_nested_oof_patient_predictions_ridge.csv",
            "FINAL_nested_oof_patient_predictions_xgboost.csv",
            "FINAL_nested_oof_window_predictions_ridge.csv",
            "FINAL_nested_oof_window_predictions_xgboost.csv",
        ]
    elif experiment_type == "cnn":
        required = [
            "FINAL_nested_oof_patient_predictions.csv",
            "FINAL_nested_oof_window_predictions.csv",
        ]
    else:
        raise ValueError(experiment_type)
    return all(os.path.isfile(os.path.join(directory, f)) for f in required)


def discover_experiment_dir(
    root: str,
    experiment_type: str,
    expected_basename: str,
    override: Optional[str] = None,
) -> str:
    if override is not None:
        path = os.path.abspath(override)
        if not _required_prediction_files(path, experiment_type):
            raise FileNotFoundError(
                f"Override does not contain all required prediction files: {path}"
            )
        return path

    exact = os.path.join(root, expected_basename)
    if _required_prediction_files(exact, experiment_type):
        return os.path.abspath(exact)

    if not os.path.isdir(root):
        raise FileNotFoundError(f"Result root not found: {root}")

    candidates: List[str] = []
    for current, dirs, files in os.walk(root):
        if _required_prediction_files(current, experiment_type):
            candidates.append(os.path.abspath(current))

    if not candidates:
        raise FileNotFoundError(
            f"No completed {experiment_type} experiment found under: {root}"
        )

    def score(path: str) -> Tuple[int, float, str]:
        name = os.path.basename(path).lower()
        expected = expected_basename.lower()
        s = 0
        if name == expected:
            s += 1000
        if expected in name or name in expected:
            s += 200
        if experiment_type == "1d" and "1d" in name:
            s += 100
        if experiment_type == "2d" and ("2d" in name or "flatten" in name):
            s += 100
        if experiment_type == "cnn" and "120min" in name:
            s += 100
        if "mean1weight" in name:
            s += 50
        if "v2" in name:
            s += 20
        mtime = os.path.getmtime(path)
        return s, mtime, path

    ranked = sorted(candidates, key=score, reverse=True)
    best = ranked[0]
    if len(ranked) > 1 and score(ranked[0])[:2] == score(ranked[1])[:2]:
        raise RuntimeError(
            "Automatic experiment discovery is ambiguous. Set the corresponding "
            "*_DIR_OVERRIDE. Candidates:\n" + "\n".join(ranked)
        )

    print(f"Auto-selected {experiment_type} experiment: {best}")
    if len(ranked) > 1:
        print("Other completed candidates were not selected:")
        for p in ranked[1:]:
            print("  -", p)
    return best


BASELINE_1D_DIR = discover_experiment_dir(
    BASELINE_ROOT, "1d", EXPECTED_1D_BASENAME, BASELINE_1D_DIR_OVERRIDE
)
BASELINE_2D_DIR = discover_experiment_dir(
    BASELINE_ROOT, "2d", EXPECTED_2D_BASENAME, BASELINE_2D_DIR_OVERRIDE
)
CNN_DIR = discover_experiment_dir(
    NESTED_ROOT, "cnn", EXPECTED_CNN_BASENAME, CNN_DIR_OVERRIDE
)

print("\nResolved directories")
print("1D baseline:", BASELINE_1D_DIR)
print("2D baseline:", BASELINE_2D_DIR)
print("CNN:", CNN_DIR)
print("Comparison outputs:", OUTPUT_DIR)


# ============================================================
# 2. Loading and column standardization
# ============================================================
def normalize_pid(value) -> str:
    text = str(value)
    if text.endswith(".0") and text[:-2].replace("-", "").isdigit():
        return text[:-2]
    return text


def first_existing(columns: Iterable[str], candidates: Sequence[str], label: str) -> str:
    columns = set(columns)
    for col in candidates:
        if col in columns:
            return col
    raise KeyError(f"Could not find {label}. Tried: {list(candidates)}")


def load_patient_predictions(path: str, model_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"pid": str})
    if "pid" not in df.columns:
        raise KeyError(f"pid column missing in {path}")

    pred_col = first_existing(
        df.columns,
        ["pred_patient", "pred_patient_mean", "prediction", "pred"],
        "patient prediction column",
    )
    true_col = first_existing(
        df.columns,
        ["true_patient", "target_patient", "true", "target"],
        "patient truth column",
    )

    out = pd.DataFrame(
        {
            "pid": df["pid"].map(normalize_pid),
            "true_patient_native": pd.to_numeric(df[true_col], errors="coerce"),
            "pred_patient": pd.to_numeric(df[pred_col], errors="coerce"),
        }
    )
    out["outer_fold"] = (
        pd.to_numeric(df["outer_fold"], errors="coerce")
        if "outer_fold" in df.columns
        else np.nan
    )
    out["model"] = model_name
    out = out.dropna(subset=["pid", "true_patient_native", "pred_patient"]).copy()

    if out["pid"].duplicated().any():
        duplicates = out.loc[out["pid"].duplicated(False), "pid"].unique()[:20]
        raise RuntimeError(
            f"{model_name}: duplicate patient rows in {path}: {duplicates.tolist()}"
        )

    out["error_patient_native"] = out["pred_patient"] - out["true_patient_native"]
    out["abs_error_patient_native"] = np.abs(out["error_patient_native"])
    return out


def load_window_predictions(path: str, model_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"pid": str})
    if "pid" not in df.columns:
        raise KeyError(f"pid column missing in {path}")

    pred_col = first_existing(
        df.columns,
        ["pred_window_mean", "pred_window", "prediction", "pred"],
        "window prediction column",
    )
    true_col = first_existing(
        df.columns,
        ["true_window", "target_window", "true", "target"],
        "window truth column",
    )

    if "window_idx" in df.columns:
        window_idx = pd.to_numeric(df["window_idx"], errors="coerce")
    elif "window_id" in df.columns:
        # Last underscore-separated token is the index in all current pipelines.
        window_idx = pd.to_numeric(
            df["window_id"].astype(str).str.rsplit("_", n=1).str[-1],
            errors="coerce",
        )
    else:
        # Safe only when the file is already ordered within each patient.
        warnings.warn(
            f"{model_name}: no window_idx/window_id found; reconstructing by row order."
        )
        window_idx = df.groupby(df["pid"].map(normalize_pid), sort=False).cumcount()

    out = pd.DataFrame(
        {
            "pid": df["pid"].map(normalize_pid),
            "window_idx": window_idx,
            "true_window": pd.to_numeric(df[true_col], errors="coerce"),
            "pred_window": pd.to_numeric(df[pred_col], errors="coerce"),
        }
    )
    out["outer_fold"] = (
        pd.to_numeric(df["outer_fold"], errors="coerce")
        if "outer_fold" in df.columns
        else np.nan
    )
    out["model"] = model_name
    out = out.dropna(subset=["pid", "window_idx", "true_window", "pred_window"]).copy()
    out["window_idx"] = out["window_idx"].astype(int)

    if out.duplicated(["pid", "window_idx"]).any():
        bad = out.loc[
            out.duplicated(["pid", "window_idx"], keep=False),
            ["pid", "window_idx"],
        ].head(20)
        raise RuntimeError(
            f"{model_name}: duplicate patient-window rows in {path}:\n{bad}"
        )

    out["window_id"] = [
        f"{pid}_{idx}" for pid, idx in zip(out["pid"], out["window_idx"])
    ]
    out["error_window"] = out["pred_window"] - out["true_window"]
    out["abs_error_window"] = np.abs(out["error_window"])
    return out


patient_files = {
    "Ridge 1D": os.path.join(BASELINE_1D_DIR, "FINAL_nested_oof_patient_predictions_ridge.csv"),
    "XGBoost 1D": os.path.join(BASELINE_1D_DIR, "FINAL_nested_oof_patient_predictions_xgboost.csv"),
    "Ridge 2D": os.path.join(BASELINE_2D_DIR, "FINAL_nested_oof_patient_predictions_ridge.csv"),
    "XGBoost 2D": os.path.join(BASELINE_2D_DIR, "FINAL_nested_oof_patient_predictions_xgboost.csv"),
    "CNN 120min": os.path.join(CNN_DIR, "FINAL_nested_oof_patient_predictions.csv"),
}
window_files = {
    "Ridge 1D": os.path.join(BASELINE_1D_DIR, "FINAL_nested_oof_window_predictions_ridge.csv"),
    "XGBoost 1D": os.path.join(BASELINE_1D_DIR, "FINAL_nested_oof_window_predictions_xgboost.csv"),
    "Ridge 2D": os.path.join(BASELINE_2D_DIR, "FINAL_nested_oof_window_predictions_ridge.csv"),
    "XGBoost 2D": os.path.join(BASELINE_2D_DIR, "FINAL_nested_oof_window_predictions_xgboost.csv"),
    "CNN 120min": os.path.join(CNN_DIR, "FINAL_nested_oof_window_predictions.csv"),
}

for path in list(patient_files.values()) + list(window_files.values()):
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

patient_predictions: Dict[str, pd.DataFrame] = {
    model: load_patient_predictions(path, model)
    for model, path in patient_files.items()
}
window_predictions: Dict[str, pd.DataFrame] = {
    model: load_window_predictions(path, model)
    for model, path in window_files.items()
}


# ============================================================
# 3. Metrics and statistical helpers
# ============================================================
def safe_r2(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    keep = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[keep], y_pred[keep]
    if len(y_true) < 2 or np.allclose(y_true, y_true[0]):
        return np.nan
    return float(r2_score(y_true, y_pred))


def regression_metrics(y_true: Sequence[float], y_pred: Sequence[float]) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    keep = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[keep], y_pred[keep]
    if len(y_true) == 0:
        return {
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "bias": np.nan,
            "n": 0,
        }
    error = y_pred - y_true
    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "r2": safe_r2(y_true, y_pred),
        "bias": float(np.mean(error)),
        "n": int(len(y_true)),
    }


def safe_correlations(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float, float, float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) < 3 or np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return np.nan, np.nan, np.nan, np.nan
    if not SCIPY_AVAILABLE:
        return float(np.corrcoef(x, y)[0, 1]), np.nan, np.nan, np.nan
    p_r, p_p = pearsonr(x, y)
    s_r, s_p = spearmanr(x, y)
    return float(p_r), float(p_p), float(s_r), float(s_p)


def safe_wilcoxon(a: Sequence[float], b: Sequence[float]) -> Tuple[float, float]:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    keep = np.isfinite(a) & np.isfinite(b)
    diff = a[keep] - b[keep]
    if not SCIPY_AVAILABLE or len(diff) == 0 or np.allclose(diff, 0.0):
        return np.nan, np.nan
    try:
        stat, p = wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan


def bootstrap_paired_differences(
    true_values: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    repeats: int = BOOTSTRAP_REPEATS,
    seed: int = BOOTSTRAP_SEED,
) -> Dict[str, float]:
    true_values = np.asarray(true_values, dtype=float)
    pred_a = np.asarray(pred_a, dtype=float)
    pred_b = np.asarray(pred_b, dtype=float)
    keep = np.isfinite(true_values) & np.isfinite(pred_a) & np.isfinite(pred_b)
    true_values, pred_a, pred_b = true_values[keep], pred_a[keep], pred_b[keep]
    n = len(true_values)
    if n == 0:
        return {
            "delta_mae": np.nan,
            "delta_mae_ci_low": np.nan,
            "delta_mae_ci_high": np.nan,
            "delta_rmse": np.nan,
            "delta_rmse_ci_low": np.nan,
            "delta_rmse_ci_high": np.nan,
        }

    abs_a = np.abs(pred_a - true_values)
    abs_b = np.abs(pred_b - true_values)
    sq_a = (pred_a - true_values) ** 2
    sq_b = (pred_b - true_values) ** 2

    observed_delta_mae = float(abs_a.mean() - abs_b.mean())
    observed_delta_rmse = float(np.sqrt(sq_a.mean()) - np.sqrt(sq_b.mean()))

    rng = np.random.default_rng(seed)
    delta_mae_boot = np.empty(repeats, dtype=float)
    delta_rmse_boot = np.empty(repeats, dtype=float)
    for i in range(repeats):
        idx = rng.integers(0, n, size=n)
        delta_mae_boot[i] = abs_a[idx].mean() - abs_b[idx].mean()
        delta_rmse_boot[i] = np.sqrt(sq_a[idx].mean()) - np.sqrt(sq_b[idx].mean())

    return {
        "delta_mae": observed_delta_mae,
        "delta_mae_ci_low": float(np.quantile(delta_mae_boot, 0.025)),
        "delta_mae_ci_high": float(np.quantile(delta_mae_boot, 0.975)),
        "delta_rmse": observed_delta_rmse,
        "delta_rmse_ci_low": float(np.quantile(delta_rmse_boot, 0.025)),
        "delta_rmse_ci_high": float(np.quantile(delta_rmse_boot, 0.975)),
    }


def bootstrap_paired_patient_scores(
    score_a: np.ndarray,
    score_b: np.ndarray,
    repeats: int = BOOTSTRAP_REPEATS,
    seed: int = BOOTSTRAP_SEED,
) -> Tuple[float, float, float]:
    """Bootstrap paired patient-level scores; negative delta means A is better."""
    score_a = np.asarray(score_a, dtype=float)
    score_b = np.asarray(score_b, dtype=float)
    keep = np.isfinite(score_a) & np.isfinite(score_b)
    score_a, score_b = score_a[keep], score_b[keep]
    if len(score_a) == 0:
        return np.nan, np.nan, np.nan
    observed = float(np.mean(score_a - score_b))
    rng = np.random.default_rng(seed)
    n = len(score_a)
    boot = np.empty(repeats, dtype=float)
    for i in range(repeats):
        idx = rng.integers(0, n, size=n)
        boot[i] = np.mean(score_a[idx] - score_b[idx])
    return observed, float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


# ============================================================
# 4. Common patient truth and wide patient prediction table
# ============================================================
cnn_patient = patient_predictions["CNN 120min"].copy()
common_truth = cnn_patient[["pid", "true_patient_native", "outer_fold"]].rename(
    columns={
        "true_patient_native": "true_patient_common_CNN",
        "outer_fold": "outer_fold_common",
    }
)

wide_patient = common_truth.copy()
for model in MODEL_ORDER:
    tmp = patient_predictions[model][["pid", "pred_patient", "true_patient_native", "outer_fold"]].copy()
    safe = model.lower().replace(" ", "_").replace("-", "_")
    tmp = tmp.rename(
        columns={
            "pred_patient": f"pred_{safe}",
            "true_patient_native": f"true_native_{safe}",
            "outer_fold": f"outer_fold_{safe}",
        }
    )
    wide_patient = wide_patient.merge(tmp, on="pid", how="outer", validate="one_to_one")

wide_patient.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_patient_predictions_wide_common_truth.csv"),
    index=False,
)


# ============================================================
# 5. Overall summary: native metrics + common patient truth
# ============================================================
summary_rows: List[Dict[str, object]] = []
for model in MODEL_ORDER:
    pat = patient_predictions[model].copy()
    win = window_predictions[model].copy()

    native_patient = regression_metrics(pat["true_patient_native"], pat["pred_patient"])
    native_window = regression_metrics(win["true_window"], win["pred_window"])

    common_pat = pat.merge(common_truth[["pid", "true_patient_common_CNN"]], on="pid", how="inner")
    common_patient_metrics = regression_metrics(
        common_pat["true_patient_common_CNN"], common_pat["pred_patient"]
    )
    truth_difference = common_pat["true_patient_native"] - common_pat["true_patient_common_CNN"]

    row = {
        "model": model,
        **MODEL_META[model],
        "n_patients": int(native_patient["n"]),
        "n_windows": int(native_window["n"]),
        "patient_mae_native": native_patient["mae"],
        "patient_rmse_native": native_patient["rmse"],
        "patient_r2_native": native_patient["r2"],
        "patient_bias_native": native_patient["bias"],
        "patient_mae_common_CNN_truth": common_patient_metrics["mae"],
        "patient_rmse_common_CNN_truth": common_patient_metrics["rmse"],
        "patient_r2_common_CNN_truth": common_patient_metrics["r2"],
        "patient_bias_common_CNN_truth": common_patient_metrics["bias"],
        "mean_native_minus_common_truth": float(truth_difference.mean()),
        "max_abs_native_minus_common_truth": float(np.abs(truth_difference).max()),
        "window_mae_native": native_window["mae"],
        "window_rmse_native": native_window["rmse"],
        "window_r2_native": native_window["r2"],
        "window_bias_native": native_window["bias"],
        "window_metrics_directly_comparable_across_all_models": False,
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df["model"] = pd.Categorical(summary_df["model"], MODEL_ORDER, ordered=True)
summary_df = summary_df.sort_values("model").reset_index(drop=True)
summary_df["model"] = summary_df["model"].astype(str)

summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_model_comparison_summary_all_metrics.csv"),
    index=False,
)

# Compact requested table, using common patient truth for a fair five-model comparison.
requested_table = summary_df[
    [
        "model",
        "input",
        "window_definition",
        "n_patients",
        "n_windows",
        "patient_mae_common_CNN_truth",
        "patient_rmse_common_CNN_truth",
        "patient_r2_common_CNN_truth",
        "window_mae_native",
        "window_rmse_native",
        "window_r2_native",
        "window_comparability_group",
    ]
].copy()
requested_table.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_requested_model_comparison_table.csv"),
    index=False,
)


# ============================================================
# 6. Outer-fold metrics
# ============================================================
fold_rows: List[Dict[str, object]] = []
for model in MODEL_ORDER:
    pat = patient_predictions[model].merge(
        common_truth[["pid", "true_patient_common_CNN", "outer_fold_common"]],
        on="pid",
        how="inner",
    )
    win = window_predictions[model].copy()

    fold_values = sorted(
        set(pd.to_numeric(pat["outer_fold_common"], errors="coerce").dropna().astype(int))
    )
    for fold in fold_values:
        p = pat[pat["outer_fold_common"] == fold]
        native_p = regression_metrics(p["true_patient_native"], p["pred_patient"])
        common_p = regression_metrics(p["true_patient_common_CNN"], p["pred_patient"])

        # Prefer model-native outer_fold in the window file; all pipelines reused the CNN folds.
        w = win[pd.to_numeric(win["outer_fold"], errors="coerce") == fold]
        native_w = regression_metrics(w["true_window"], w["pred_window"])

        fold_rows.append(
            {
                "model": model,
                "outer_fold": int(fold),
                "n_patients": int(len(p)),
                "n_windows": int(len(w)),
                "patient_mae_native": native_p["mae"],
                "patient_rmse_native": native_p["rmse"],
                "patient_r2_native": native_p["r2"],
                "patient_mae_common_CNN_truth": common_p["mae"],
                "patient_rmse_common_CNN_truth": common_p["rmse"],
                "patient_r2_common_CNN_truth": common_p["r2"],
                "window_mae_native": native_w["mae"],
                "window_rmse_native": native_w["rmse"],
                "window_r2_native": native_w["r2"],
                "window_comparability_group": MODEL_META[model]["window_comparability_group"],
            }
        )

fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_outer_fold_metrics_all_models.csv"),
    index=False,
)

fold_summary = (
    fold_df.groupby("model", as_index=False)
    .agg(
        n_outer_folds=("outer_fold", "nunique"),
        patient_mae_common_mean=("patient_mae_common_CNN_truth", "mean"),
        patient_mae_common_std=("patient_mae_common_CNN_truth", lambda x: float(np.std(x, ddof=0))),
        patient_rmse_common_mean=("patient_rmse_common_CNN_truth", "mean"),
        patient_rmse_common_std=("patient_rmse_common_CNN_truth", lambda x: float(np.std(x, ddof=0))),
        window_mae_native_mean=("window_mae_native", "mean"),
        window_mae_native_std=("window_mae_native", lambda x: float(np.std(x, ddof=0))),
    )
)
fold_summary.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_outer_fold_metrics_mean_std.csv"),
    index=False,
)


# ============================================================
# 7. Pairwise patient-level comparisons among all five models
#    Uses one common truth: CNN's patient-level true PMA.
# ============================================================
pairwise_patient_rows: List[Dict[str, object]] = []
for pair_index, (model_a, model_b) in enumerate(itertools.combinations(MODEL_ORDER, 2)):
    a = patient_predictions[model_a][["pid", "pred_patient"]].rename(
        columns={"pred_patient": "pred_a"}
    )
    b = patient_predictions[model_b][["pid", "pred_patient"]].rename(
        columns={"pred_patient": "pred_b"}
    )
    paired = common_truth[["pid", "true_patient_common_CNN"]].merge(a, on="pid", how="inner")
    paired = paired.merge(b, on="pid", how="inner")

    paired["abs_error_a"] = np.abs(paired["pred_a"] - paired["true_patient_common_CNN"])
    paired["abs_error_b"] = np.abs(paired["pred_b"] - paired["true_patient_common_CNN"])
    paired["sq_error_a"] = (paired["pred_a"] - paired["true_patient_common_CNN"]) ** 2
    paired["sq_error_b"] = (paired["pred_b"] - paired["true_patient_common_CNN"]) ** 2

    boot = bootstrap_paired_differences(
        paired["true_patient_common_CNN"].to_numpy(),
        paired["pred_a"].to_numpy(),
        paired["pred_b"].to_numpy(),
        seed=BOOTSTRAP_SEED + pair_index,
    )
    wilcoxon_stat, wilcoxon_p = safe_wilcoxon(
        paired["abs_error_a"], paired["abs_error_b"]
    )
    pearson_r, pearson_p, spearman_r, spearman_p = safe_correlations(
        paired["pred_a"], paired["pred_b"]
    )

    delta_abs = paired["abs_error_a"] - paired["abs_error_b"]
    pairwise_patient_rows.append(
        {
            "model_a": model_a,
            "model_b": model_b,
            "interpretation": "negative delta means model_a is better",
            "n_common_patients": int(len(paired)),
            "mae_a": float(paired["abs_error_a"].mean()),
            "mae_b": float(paired["abs_error_b"].mean()),
            **boot,
            "win_rate_a": float(np.mean(delta_abs < -1e-12)),
            "tie_rate": float(np.mean(np.abs(delta_abs) <= 1e-12)),
            "win_rate_b": float(np.mean(delta_abs > 1e-12)),
            "median_abs_error_difference_a_minus_b": float(np.median(delta_abs)),
            "wilcoxon_statistic_abs_error": wilcoxon_stat,
            "wilcoxon_p_value_abs_error": wilcoxon_p,
            "prediction_pearson_r": pearson_r,
            "prediction_pearson_p": pearson_p,
            "prediction_spearman_r": spearman_r,
            "prediction_spearman_p": spearman_p,
        }
    )

pairwise_patient_df = pd.DataFrame(pairwise_patient_rows)
pairwise_patient_df.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_pairwise_patient_comparison_common_CNN_truth.csv"),
    index=False,
)


# ============================================================
# 8. Window-level comparison
#
# All models receive native window metrics in the main summary.
# Paired tests are restricted to identical window definitions:
#   - Ridge 1D vs XGBoost 1D
#   - Ridge 2D vs XGBoost 2D vs CNN 120min
#
# Repeated windows from the same patient are correlated, so confidence intervals
# and Wilcoxon tests use one aggregated window-error score per patient.
# ============================================================
window_groups = {
    "1D_10min": ["Ridge 1D", "XGBoost 1D"],
    "2D_120min": ["Ridge 2D", "XGBoost 2D", "CNN 120min"],
}

window_pair_rows: List[Dict[str, object]] = []
for group_name, models in window_groups.items():
    # Save a wide exact-window table for optional downstream analysis.
    base_model = models[0]
    wide = window_predictions[base_model][
        ["pid", "window_idx", "window_id", "true_window", "outer_fold", "pred_window"]
    ].rename(columns={"pred_window": f"pred_{base_model.lower().replace(' ', '_')}"})

    for model in models[1:]:
        tmp = window_predictions[model][
            ["pid", "window_idx", "true_window", "pred_window"]
        ].rename(
            columns={
                "true_window": f"true_{model.lower().replace(' ', '_')}",
                "pred_window": f"pred_{model.lower().replace(' ', '_')}",
            }
        )
        wide = wide.merge(tmp, on=["pid", "window_idx"], how="inner")

    wide.to_csv(
        os.path.join(OUTPUT_DIR, f"FINAL_window_predictions_wide_{group_name}.csv"),
        index=False,
    )

    for pair_index, (model_a, model_b) in enumerate(itertools.combinations(models, 2)):
        a = window_predictions[model_a][
            ["pid", "window_idx", "true_window", "pred_window"]
        ].rename(columns={"true_window": "true_a", "pred_window": "pred_a"})
        b = window_predictions[model_b][
            ["pid", "window_idx", "true_window", "pred_window"]
        ].rename(columns={"true_window": "true_b", "pred_window": "pred_b"})
        paired = a.merge(b, on=["pid", "window_idx"], how="inner", validate="one_to_one")

        n_a, n_b = len(a), len(b)
        n_common = len(paired)
        max_true_diff = float(np.max(np.abs(paired["true_a"] - paired["true_b"]))) if n_common else np.nan
        complete_alignment = n_common == n_a == n_b and max_true_diff <= TRUE_TOLERANCE

        if n_common == 0:
            window_pair_rows.append(
                {
                    "comparability_group": group_name,
                    "model_a": model_a,
                    "model_b": model_b,
                    "status": "skipped_no_common_windows",
                    "n_windows_a": n_a,
                    "n_windows_b": n_b,
                    "n_common_windows": 0,
                }
            )
            continue

        if max_true_diff > TRUE_TOLERANCE:
            warnings.warn(
                f"{model_a} vs {model_b}: matched window truths differ by up to "
                f"{max_true_diff:.6g}. Pairwise window inference is skipped."
            )
            window_pair_rows.append(
                {
                    "comparability_group": group_name,
                    "model_a": model_a,
                    "model_b": model_b,
                    "status": "skipped_truth_mismatch",
                    "n_windows_a": n_a,
                    "n_windows_b": n_b,
                    "n_common_windows": n_common,
                    "max_abs_true_difference": max_true_diff,
                }
            )
            continue

        paired["abs_error_a"] = np.abs(paired["pred_a"] - paired["true_a"])
        paired["abs_error_b"] = np.abs(paired["pred_b"] - paired["true_a"])
        paired["sq_error_a"] = (paired["pred_a"] - paired["true_a"]) ** 2
        paired["sq_error_b"] = (paired["pred_b"] - paired["true_a"]) ** 2

        # Patient-clustered window error scores.
        patient_scores = (
            paired.groupby("pid", as_index=False)
            .agg(
                n_common_windows=("window_idx", "size"),
                patient_mean_abs_window_error_a=("abs_error_a", "mean"),
                patient_mean_abs_window_error_b=("abs_error_b", "mean"),
                patient_mean_sq_window_error_a=("sq_error_a", "mean"),
                patient_mean_sq_window_error_b=("sq_error_b", "mean"),
            )
        )
        patient_scores["patient_window_rmse_a"] = np.sqrt(
            patient_scores["patient_mean_sq_window_error_a"]
        )
        patient_scores["patient_window_rmse_b"] = np.sqrt(
            patient_scores["patient_mean_sq_window_error_b"]
        )

        delta_patient_mae, delta_patient_mae_low, delta_patient_mae_high = (
            bootstrap_paired_patient_scores(
                patient_scores["patient_mean_abs_window_error_a"].to_numpy(),
                patient_scores["patient_mean_abs_window_error_b"].to_numpy(),
                seed=BOOTSTRAP_SEED + 100 + pair_index,
            )
        )
        delta_patient_rmse, delta_patient_rmse_low, delta_patient_rmse_high = (
            bootstrap_paired_patient_scores(
                patient_scores["patient_window_rmse_a"].to_numpy(),
                patient_scores["patient_window_rmse_b"].to_numpy(),
                seed=BOOTSTRAP_SEED + 200 + pair_index,
            )
        )
        wilcoxon_stat, wilcoxon_p = safe_wilcoxon(
            patient_scores["patient_mean_abs_window_error_a"],
            patient_scores["patient_mean_abs_window_error_b"],
        )
        pearson_r, pearson_p, spearman_r, spearman_p = safe_correlations(
            paired["pred_a"], paired["pred_b"]
        )

        score_diff = (
            patient_scores["patient_mean_abs_window_error_a"]
            - patient_scores["patient_mean_abs_window_error_b"]
        )
        pooled_a = regression_metrics(paired["true_a"], paired["pred_a"])
        pooled_b = regression_metrics(paired["true_a"], paired["pred_b"])

        window_pair_rows.append(
            {
                "comparability_group": group_name,
                "model_a": model_a,
                "model_b": model_b,
                "status": "complete_alignment" if complete_alignment else "partial_alignment",
                "interpretation": "negative delta means model_a is better",
                "n_windows_a": n_a,
                "n_windows_b": n_b,
                "n_common_windows": n_common,
                "n_common_patients": int(patient_scores["pid"].nunique()),
                "common_fraction_a": float(n_common / n_a) if n_a else np.nan,
                "common_fraction_b": float(n_common / n_b) if n_b else np.nan,
                "max_abs_true_difference": max_true_diff,
                "pooled_window_mae_a": pooled_a["mae"],
                "pooled_window_mae_b": pooled_b["mae"],
                "pooled_window_mae_delta_a_minus_b": pooled_a["mae"] - pooled_b["mae"],
                "pooled_window_rmse_a": pooled_a["rmse"],
                "pooled_window_rmse_b": pooled_b["rmse"],
                "pooled_window_rmse_delta_a_minus_b": pooled_a["rmse"] - pooled_b["rmse"],
                "patient_balanced_mean_window_mae_a": float(
                    patient_scores["patient_mean_abs_window_error_a"].mean()
                ),
                "patient_balanced_mean_window_mae_b": float(
                    patient_scores["patient_mean_abs_window_error_b"].mean()
                ),
                "patient_balanced_window_mae_delta_a_minus_b": delta_patient_mae,
                "patient_balanced_window_mae_delta_ci_low": delta_patient_mae_low,
                "patient_balanced_window_mae_delta_ci_high": delta_patient_mae_high,
                "patient_balanced_mean_window_rmse_a": float(
                    patient_scores["patient_window_rmse_a"].mean()
                ),
                "patient_balanced_mean_window_rmse_b": float(
                    patient_scores["patient_window_rmse_b"].mean()
                ),
                "patient_balanced_window_rmse_delta_a_minus_b": delta_patient_rmse,
                "patient_balanced_window_rmse_delta_ci_low": delta_patient_rmse_low,
                "patient_balanced_window_rmse_delta_ci_high": delta_patient_rmse_high,
                "patient_win_rate_a": float(np.mean(score_diff < -1e-12)),
                "patient_tie_rate": float(np.mean(np.abs(score_diff) <= 1e-12)),
                "patient_win_rate_b": float(np.mean(score_diff > 1e-12)),
                "wilcoxon_statistic_patient_mean_abs_window_error": wilcoxon_stat,
                "wilcoxon_p_value_patient_mean_abs_window_error": wilcoxon_p,
                "window_prediction_pearson_r": pearson_r,
                "window_prediction_pearson_p": pearson_p,
                "window_prediction_spearman_r": spearman_r,
                "window_prediction_spearman_p": spearman_p,
            }
        )

window_pair_df = pd.DataFrame(window_pair_rows)
window_pair_df.to_csv(
    os.path.join(OUTPUT_DIR, "FINAL_pairwise_window_comparison_compatible_models.csv"),
    index=False,
)


# ============================================================
# 9. Optional GA-group comparison
#    Reads existing GA-analysis files when available; does not require them.
# ============================================================
def locate_ga_summary(experiment_dir: str, model: Optional[str] = None) -> Optional[str]:
    candidates: List[str] = []
    if model is not None:
        candidates.extend(
            [
                os.path.join(experiment_dir, "ga_bin_analysis", model, "ga_bin_group_summary.csv"),
                os.path.join(experiment_dir, "ga_bin_analysis", model.lower(), "ga_bin_group_summary.csv"),
            ]
        )
    else:
        candidates.extend(
            [
                os.path.join(experiment_dir, "ga_bin_analysis", "ga_bin_group_summary.csv"),
                os.path.join(experiment_dir, "ga_bin_analysis", "ga_bin_panel_summary.csv"),
            ]
        )
    for path in candidates:
        if os.path.isfile(path):
            return path
    return None


ga_sources = {
    "Ridge 1D": locate_ga_summary(BASELINE_1D_DIR, "ridge"),
    "XGBoost 1D": locate_ga_summary(BASELINE_1D_DIR, "xgboost"),
    "Ridge 2D": locate_ga_summary(BASELINE_2D_DIR, "ridge"),
    "XGBoost 2D": locate_ga_summary(BASELINE_2D_DIR, "xgboost"),
    "CNN 120min": locate_ga_summary(CNN_DIR, None),
}

ga_frames: List[pd.DataFrame] = []
for model, path in ga_sources.items():
    if path is None:
        continue
    ga = pd.read_csv(path)
    ga["model"] = model
    ga["source_file"] = path
    ga_frames.append(ga)

if ga_frames:
    ga_comparison = pd.concat(ga_frames, ignore_index=True, sort=False)
    ga_comparison.to_csv(
        os.path.join(OUTPUT_DIR, "FINAL_GA_group_model_comparison_if_available.csv"),
        index=False,
    )
else:
    ga_comparison = pd.DataFrame()
    print("No GA-group summary files were found; optional GA comparison skipped.")


# ============================================================
# 10. Figures
# ============================================================
def save_patient_metric_plot(summary: pd.DataFrame) -> None:
    metrics = [
        ("patient_mae_common_CNN_truth", "Patient MAE (weeks)"),
        ("patient_rmse_common_CNN_truth", "Patient RMSE (weeks)"),
        ("patient_r2_common_CNN_truth", "Patient R²"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    x = np.arange(len(summary))
    for ax, (column, ylabel) in zip(axes, metrics):
        values = summary[column].to_numpy(dtype=float)
        ax.bar(x, values)
        ax.set_xticks(x)
        ax.set_xticklabels(summary["model"], rotation=35, ha="right")
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.25)
        for i, value in enumerate(values):
            if np.isfinite(value):
                ax.text(i, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
    fig.suptitle("Patient-level comparison using common CNN patient truth")
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "01_patient_metrics_common_CNN_truth.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_window_metric_plot(summary: pd.DataFrame) -> None:
    metrics = [
        ("window_mae_native", "Window MAE (weeks)"),
        ("window_rmse_native", "Window RMSE (weeks)"),
        ("window_r2_native", "Window R²"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    x = np.arange(len(summary))
    for ax, (column, ylabel) in zip(axes, metrics):
        values = summary[column].to_numpy(dtype=float)
        ax.bar(x, values)
        ax.set_xticks(x)
        ax.set_xticklabels(summary["model"], rotation=35, ha="right")
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.25)
        for i, value in enumerate(values):
            if np.isfinite(value):
                ax.text(i, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
    fig.suptitle(
        "Native window-level metrics (10-min and 120-min groups are not directly paired)"
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "02_window_metrics_native_all_models.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_fold_plot(fold_metrics: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(11, 6))
    for model in MODEL_ORDER:
        sub = fold_metrics[fold_metrics["model"] == model].sort_values("outer_fold")
        if sub.empty:
            continue
        ax.plot(
            sub["outer_fold"],
            sub["patient_mae_common_CNN_truth"],
            marker="o",
            linewidth=2,
            label=model,
        )
    ax.set_xlabel("Outer fold")
    ax.set_ylabel("Patient MAE using common CNN truth (weeks)")
    ax.set_xticks(sorted(fold_metrics["outer_fold"].dropna().unique()))
    ax.grid(True, alpha=0.25)
    ax.legend()
    ax.set_title("Outer-fold patient MAE comparison")
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "03_outer_fold_patient_MAE_common_truth.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_pairwise_heatmap(pairwise: pd.DataFrame) -> None:
    matrix = pd.DataFrame(0.0, index=MODEL_ORDER, columns=MODEL_ORDER)
    matrix.loc[:, :] = np.nan
    np.fill_diagonal(matrix.values, 0.0)
    for _, row in pairwise.iterrows():
        a, b, delta = row["model_a"], row["model_b"], row["delta_mae"]
        matrix.loc[a, b] = delta
        matrix.loc[b, a] = -delta

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto")
    ax.set_xticks(np.arange(len(MODEL_ORDER)))
    ax.set_yticks(np.arange(len(MODEL_ORDER)))
    ax.set_xticklabels(MODEL_ORDER, rotation=40, ha="right")
    ax.set_yticklabels(MODEL_ORDER)
    for i in range(len(MODEL_ORDER)):
        for j in range(len(MODEL_ORDER)):
            value = matrix.iloc[i, j]
            if np.isfinite(value):
                ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=8)
    ax.set_title("Paired patient MAE difference: row model − column model\nNegative = row model better")
    fig.colorbar(im, ax=ax, label="ΔMAE (weeks)")
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "04_pairwise_patient_delta_MAE_heatmap.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_patient_error_boxplot() -> None:
    records = []
    for model in MODEL_ORDER:
        tmp = patient_predictions[model][["pid", "pred_patient"]].merge(
            common_truth[["pid", "true_patient_common_CNN"]], on="pid", how="inner"
        )
        tmp["abs_error"] = np.abs(tmp["pred_patient"] - tmp["true_patient_common_CNN"])
        tmp["model"] = model
        records.append(tmp[["model", "abs_error"]])
    long = pd.concat(records, ignore_index=True)
    arrays = [long.loc[long["model"] == model, "abs_error"].to_numpy() for model in MODEL_ORDER]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.boxplot(arrays, labels=MODEL_ORDER, showfliers=False)
    ax.set_ylabel("Absolute patient error using common CNN truth (weeks)")
    ax.set_xticklabels(MODEL_ORDER, rotation=35, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ax.set_title("Distribution of paired patient-level absolute errors")
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "05_patient_absolute_error_boxplot.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


save_patient_metric_plot(summary_df)
save_window_metric_plot(summary_df)
save_fold_plot(fold_df)
save_pairwise_heatmap(pairwise_patient_df)
save_patient_error_boxplot()

if not ga_comparison.empty and {"ga_group", "patient_mae"}.issubset(ga_comparison.columns):
    fig, ax = plt.subplots(figsize=(12, 6))
    ga_order = ["EP < 26 wk", "VP [26,29) wk", "MP [29,32) wk", "LP [32,36) wk"]
    x = np.arange(len(ga_order))
    width = 0.15
    for i, model in enumerate(MODEL_ORDER):
        sub = ga_comparison[ga_comparison["model"] == model].set_index("ga_group")
        values = [sub.loc[g, "patient_mae"] if g in sub.index else np.nan for g in ga_order]
        ax.bar(x + (i - 2) * width, values, width=width, label=model)
    ax.set_xticks(x)
    ax.set_xticklabels(ga_order)
    ax.set_ylabel("Patient MAE (weeks)")
    ax.set_title("GA-group patient MAE comparison (available analyses)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "06_GA_group_patient_MAE_comparison_if_available.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


# ============================================================
# 11. Machine-readable manifest and console summary
# ============================================================
manifest = {
    "baseline_root": BASELINE_ROOT,
    "nested_root": NESTED_ROOT,
    "baseline_1d_dir": BASELINE_1D_DIR,
    "baseline_2d_dir": BASELINE_2D_DIR,
    "cnn_dir": CNN_DIR,
    "output_dir": OUTPUT_DIR,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "common_patient_truth": "CNN 120min patient-level true PMA",
    "window_comparison_policy": {
        "all_models": "native pooled window metrics are reported",
        "paired_inference": [
            "Ridge 1D vs XGBoost 1D",
            "Ridge 2D vs XGBoost 2D vs CNN 120min",
        ],
        "reason": "10-min and 120-min windows are different observational units",
        "cluster_handling": "window-level paired inference aggregates error per patient",
    },
    "model_files": {
        model: {
            "patient": patient_files[model],
            "window": window_files[model],
        }
        for model in MODEL_ORDER
    },
}
with open(os.path.join(OUTPUT_DIR, "comparison_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 120)
print("FINAL DETERMINISTIC MODEL COMPARISON COMPLETE")
print("=" * 120)
print("\nRequested comparison table:")
print(
    requested_table[
        [
            "model",
            "window_definition",
            "patient_mae_common_CNN_truth",
            "patient_rmse_common_CNN_truth",
            "patient_r2_common_CNN_truth",
            "window_mae_native",
            "window_rmse_native",
            "window_r2_native",
        ]
    ].to_string(index=False)
)
print("\nImportant: window metrics from 10-min and 120-min inputs are shown together for reference,")
print("but paired window-level statistical comparisons are only performed within matching window groups.")
print("\nSaved outputs to:", OUTPUT_DIR)
print("=" * 120)

In [ ]:
# ============================================================
# Publication-ready visualizations for the completed deterministic
# model-comparison analysis.
#
# Paste/run this cell immediately after:
#     final_compare_1d_2d_cnn_cell.py
#
# This cell does NOT retrain or re-evaluate the models. It reads the
# comparison CSV files already saved under result_comparison.
# ============================================================

from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# 0. Configuration
# ============================================================
PROJECT_ROOT = globals().get("PROJECT_ROOT", "/mnt/home/qinqiu/thesis")
OUTPUT_DIR = globals().get(
    "OUTPUT_DIR",
    os.path.join(PROJECT_ROOT, "result_comparison", "deterministic_models"),
)
VIS_DIR = os.path.join(OUTPUT_DIR, "final_visualizations")
os.makedirs(VIS_DIR, exist_ok=True)

MODEL_ORDER = [
    "Ridge 1D",
    "XGBoost 1D",
    "Ridge 2D",
    "XGBoost 2D",
    "CNN 120min",
]

MODEL_TO_SAFE = {
    "Ridge 1D": "ridge_1d",
    "XGBoost 1D": "xgboost_1d",
    "Ridge 2D": "ridge_2d",
    "XGBoost 2D": "xgboost_2d",
    "CNN 120min": "cnn_120min",
}

WINDOW_GROUPS = {
    "1D_10min": ["Ridge 1D", "XGBoost 1D"],
    "2D_120min": ["Ridge 2D", "XGBoost 2D", "CNN 120min"],
}

DPI = 300


# ============================================================
# 1. Input files generated by the comparison cell
# ============================================================
PATHS = {
    "summary": os.path.join(
        OUTPUT_DIR, "FINAL_requested_model_comparison_table.csv"
    ),
    "summary_all": os.path.join(
        OUTPUT_DIR, "FINAL_model_comparison_summary_all_metrics.csv"
    ),
    "fold": os.path.join(
        OUTPUT_DIR, "FINAL_outer_fold_metrics_all_models.csv"
    ),
    "fold_summary": os.path.join(
        OUTPUT_DIR, "FINAL_outer_fold_metrics_mean_std.csv"
    ),
    "pair_patient": os.path.join(
        OUTPUT_DIR,
        "FINAL_pairwise_patient_comparison_common_CNN_truth.csv",
    ),
    "pair_window": os.path.join(
        OUTPUT_DIR,
        "FINAL_pairwise_window_comparison_compatible_models.csv",
    ),
    "wide_patient": os.path.join(
        OUTPUT_DIR, "FINAL_patient_predictions_wide_common_truth.csv"
    ),
    "wide_window_1d": os.path.join(
        OUTPUT_DIR, "FINAL_window_predictions_wide_1D_10min.csv"
    ),
    "wide_window_2d": os.path.join(
        OUTPUT_DIR, "FINAL_window_predictions_wide_2D_120min.csv"
    ),
    "ga": os.path.join(
        OUTPUT_DIR, "FINAL_GA_group_model_comparison_if_available.csv"
    ),
}

required_keys = [
    "summary",
    "fold",
    "pair_patient",
    "pair_window",
    "wide_patient",
]
missing_required = [PATHS[k] for k in required_keys if not os.path.isfile(PATHS[k])]
if missing_required:
    raise FileNotFoundError(
        "Run the final comparison cell first. Missing files:\n"
        + "\n".join(missing_required)
    )

summary = pd.read_csv(PATHS["summary"])
summary_all = (
    pd.read_csv(PATHS["summary_all"])
    if os.path.isfile(PATHS["summary_all"])
    else summary.copy()
)
fold = pd.read_csv(PATHS["fold"])
fold_summary = (
    pd.read_csv(PATHS["fold_summary"])
    if os.path.isfile(PATHS["fold_summary"])
    else pd.DataFrame()
)
pair_patient = pd.read_csv(PATHS["pair_patient"])
pair_window = pd.read_csv(PATHS["pair_window"])
wide_patient = pd.read_csv(PATHS["wide_patient"], dtype={"pid": str})

for frame_name, frame in {
    "summary": summary,
    "fold": fold,
}.items():
    if "model" not in frame.columns:
        raise KeyError(f"{frame_name} is missing column 'model'.")

summary["model"] = pd.Categorical(summary["model"], MODEL_ORDER, ordered=True)
summary = summary.sort_values("model").reset_index(drop=True)
summary["model"] = summary["model"].astype(str)

fold["model"] = pd.Categorical(fold["model"], MODEL_ORDER, ordered=True)
fold = fold.sort_values(["model", "outer_fold"]).reset_index(drop=True)
fold["model"] = fold["model"].astype(str)


# ============================================================
# 2. Helpers
# ============================================================
def require_columns(df: pd.DataFrame, columns: Sequence[str], name: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")


def save_figure(fig: plt.Figure, filename: str) -> str:
    path = os.path.join(VIS_DIR, filename)
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    return path


def finite_xy(x: Iterable[float], y: Iterable[float]) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(list(x), dtype=float)
    y = np.asarray(list(y), dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    return x[keep], y[keep]


def significance_text(p: float) -> str:
    if not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "p<0.001"
    return f"p={p:.3f}"


def oriented_pair_row(
    pair_df: pd.DataFrame,
    desired_a: str,
    desired_b: str,
    delta_col: str,
    low_col: str,
    high_col: str,
) -> Optional[Dict[str, float]]:
    """Return desired_a - desired_b, reversing a stored pair when necessary."""
    direct = pair_df[
        (pair_df["model_a"] == desired_a)
        & (pair_df["model_b"] == desired_b)
    ]
    if len(direct) == 1:
        row = direct.iloc[0]
        return {
            "delta": float(row[delta_col]),
            "low": float(row[low_col]),
            "high": float(row[high_col]),
            "p": float(row.get("wilcoxon_p_value_abs_error", np.nan)),
            "win_rate_a": float(row.get("win_rate_a", np.nan)),
            "win_rate_b": float(row.get("win_rate_b", np.nan)),
        }

    reverse = pair_df[
        (pair_df["model_a"] == desired_b)
        & (pair_df["model_b"] == desired_a)
    ]
    if len(reverse) == 1:
        row = reverse.iloc[0]
        return {
            "delta": -float(row[delta_col]),
            "low": -float(row[high_col]),
            "high": -float(row[low_col]),
            "p": float(row.get("wilcoxon_p_value_abs_error", np.nan)),
            "win_rate_a": float(row.get("win_rate_b", np.nan)),
            "win_rate_b": float(row.get("win_rate_a", np.nan)),
        }

    return None


def prediction_column(model: str) -> str:
    return f"pred_{MODEL_TO_SAFE[model]}"


def patient_error_long_table() -> pd.DataFrame:
    require_columns(
        wide_patient,
        ["pid", "true_patient_common_CNN"]
        + [prediction_column(m) for m in MODEL_ORDER],
        "wide patient predictions",
    )

    pieces = []
    for model in MODEL_ORDER:
        pred_col = prediction_column(model)
        tmp = wide_patient[["pid", "true_patient_common_CNN", pred_col]].copy()
        tmp = tmp.rename(columns={pred_col: "prediction"})
        tmp["model"] = model
        tmp["error"] = tmp["prediction"] - tmp["true_patient_common_CNN"]
        tmp["abs_error"] = np.abs(tmp["error"])
        tmp["sq_error"] = tmp["error"] ** 2
        pieces.append(tmp)

    return pd.concat(pieces, ignore_index=True)


patient_error_long = patient_error_long_table()
created_files: List[Dict[str, str]] = []


# ============================================================
# 3. Patient-level headline metrics
# ============================================================
def plot_metric_bar(
    column: str,
    ylabel: str,
    title: str,
    filename: str,
    higher_is_better: bool = False,
) -> None:
    require_columns(summary, ["model", column], "summary")
    values = pd.to_numeric(summary[column], errors="coerce").to_numpy(dtype=float)
    x = np.arange(len(summary))

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(x, values)
    ax.set_xticks(x)
    ax.set_xticklabels(summary["model"], rotation=30, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)

    finite_values = values[np.isfinite(values)]
    if len(finite_values):
        best_value = (
            np.max(finite_values) if higher_is_better else np.min(finite_values)
        )
        for i, (bar, value) in enumerate(zip(bars, values)):
            if not np.isfinite(value):
                continue
            label = f"{value:.3f}"
            if np.isclose(value, best_value):
                label += "\n(best)"
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                label,
                ha="center",
                va="bottom",
                fontsize=9,
            )

    path = save_figure(fig, filename)
    created_files.append({"figure": filename, "description": title, "path": path})


plot_metric_bar(
    "patient_mae_common_CNN_truth",
    "Patient MAE (weeks)",
    "Patient-level MAE using a common CNN patient truth",
    "01_patient_MAE_common_truth.png",
)
plot_metric_bar(
    "patient_rmse_common_CNN_truth",
    "Patient RMSE (weeks)",
    "Patient-level RMSE using a common CNN patient truth",
    "02_patient_RMSE_common_truth.png",
)
plot_metric_bar(
    "patient_r2_common_CNN_truth",
    "Patient R²",
    "Patient-level R² using a common CNN patient truth",
    "03_patient_R2_common_truth.png",
    higher_is_better=True,
)


# ============================================================
# 4. Outer-fold stability
# ============================================================
require_columns(
    fold,
    ["model", "outer_fold", "patient_mae_common_CNN_truth"],
    "outer-fold table",
)

fig, ax = plt.subplots(figsize=(11, 6))
for model in MODEL_ORDER:
    sub = fold[fold["model"] == model].sort_values("outer_fold")
    if len(sub) == 0:
        continue
    ax.plot(
        sub["outer_fold"],
        sub["patient_mae_common_CNN_truth"],
        marker="o",
        linewidth=1.8,
        label=model,
    )
ax.set_xlabel("Outer fold")
ax.set_ylabel("Patient MAE (weeks)")
ax.set_title("Patient-level MAE across outer folds")
ax.set_xticks(sorted(pd.to_numeric(fold["outer_fold"], errors="coerce").dropna().unique()))
ax.grid(True, alpha=0.25)
ax.legend()
path = save_figure(fig, "04_outer_fold_patient_MAE_lines.png")
created_files.append(
    {
        "figure": "04_outer_fold_patient_MAE_lines.png",
        "description": "Patient-level MAE for every outer fold",
        "path": path,
    }
)

# Mean ± SD computed from the five outer folds.
fold_mae_stats = (
    fold.groupby("model", as_index=False)
    .agg(
        mean_mae=("patient_mae_common_CNN_truth", "mean"),
        sd_mae=(
            "patient_mae_common_CNN_truth",
            lambda x: float(np.std(pd.to_numeric(x, errors="coerce"), ddof=0)),
        ),
    )
)
fold_mae_stats["model"] = pd.Categorical(
    fold_mae_stats["model"], MODEL_ORDER, ordered=True
)
fold_mae_stats = fold_mae_stats.sort_values("model")

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(fold_mae_stats))
ax.errorbar(
    x,
    fold_mae_stats["mean_mae"],
    yerr=fold_mae_stats["sd_mae"],
    marker="o",
    linestyle="none",
    capsize=5,
)
ax.set_xticks(x)
ax.set_xticklabels(fold_mae_stats["model"].astype(str), rotation=30, ha="right")
ax.set_ylabel("Outer-fold patient MAE: mean ± SD (weeks)")
ax.set_title("Outer-fold stability of patient MAE")
ax.grid(axis="y", alpha=0.25)
for i, row in fold_mae_stats.reset_index(drop=True).iterrows():
    ax.text(
        i,
        row["mean_mae"] + row["sd_mae"],
        f"{row['mean_mae']:.3f} ± {row['sd_mae']:.3f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
path = save_figure(fig, "05_outer_fold_patient_MAE_mean_SD.png")
created_files.append(
    {
        "figure": "05_outer_fold_patient_MAE_mean_SD.png",
        "description": "Mean and standard deviation of patient MAE across outer folds",
        "path": path,
    }
)


# ============================================================
# 5. Pairwise patient comparison: heatmap and forest plots
# ============================================================
require_columns(
    pair_patient,
    [
        "model_a",
        "model_b",
        "delta_mae",
        "delta_mae_ci_low",
        "delta_mae_ci_high",
        "wilcoxon_p_value_abs_error",
    ],
    "patient pairwise table",
)

# Heatmap: row model - column model. Negative means row model is better.
delta_matrix = pd.DataFrame(np.nan, index=MODEL_ORDER, columns=MODEL_ORDER)
p_matrix = pd.DataFrame(np.nan, index=MODEL_ORDER, columns=MODEL_ORDER)
np.fill_diagonal(delta_matrix.values, 0.0)
for _, row in pair_patient.iterrows():
    a = str(row["model_a"])
    b = str(row["model_b"])
    if a not in MODEL_ORDER or b not in MODEL_ORDER:
        continue
    delta = float(row["delta_mae"])
    p = float(row["wilcoxon_p_value_abs_error"])
    delta_matrix.loc[a, b] = delta
    delta_matrix.loc[b, a] = -delta
    p_matrix.loc[a, b] = p
    p_matrix.loc[b, a] = p

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(delta_matrix.to_numpy(dtype=float), aspect="auto")
ax.set_xticks(np.arange(len(MODEL_ORDER)))
ax.set_yticks(np.arange(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER, rotation=35, ha="right")
ax.set_yticklabels(MODEL_ORDER)
for i in range(len(MODEL_ORDER)):
    for j in range(len(MODEL_ORDER)):
        delta = delta_matrix.iloc[i, j]
        if not np.isfinite(delta):
            continue
        if i == j:
            text = "—"
        else:
            p = p_matrix.iloc[i, j]
            marker = "*" if np.isfinite(p) and p < 0.05 else ""
            text = f"{delta:.3f}{marker}"
        ax.text(j, i, text, ha="center", va="center", fontsize=9)
ax.set_title(
    "Paired patient ΔMAE: row model − column model\n"
    "Negative = row model better; * Wilcoxon p < 0.05"
)
fig.colorbar(im, ax=ax, label="ΔMAE (weeks)")
path = save_figure(fig, "06_pairwise_patient_delta_MAE_heatmap.png")
created_files.append(
    {
        "figure": "06_pairwise_patient_delta_MAE_heatmap.png",
        "description": "Pairwise patient MAE difference heatmap",
        "path": path,
    }
)

# All 10 patient-level pairs as a forest plot.
forest_rows = []
for _, row in pair_patient.iterrows():
    forest_rows.append(
        {
            "label": f"{row['model_a']} − {row['model_b']}",
            "delta": float(row["delta_mae"]),
            "low": float(row["delta_mae_ci_low"]),
            "high": float(row["delta_mae_ci_high"]),
            "p": float(row["wilcoxon_p_value_abs_error"]),
        }
    )
forest_df = pd.DataFrame(forest_rows).sort_values("delta").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, max(6, 0.55 * len(forest_df) + 2)))
y = np.arange(len(forest_df))
xerr = np.vstack(
    [
        forest_df["delta"].to_numpy() - forest_df["low"].to_numpy(),
        forest_df["high"].to_numpy() - forest_df["delta"].to_numpy(),
    ]
)
ax.errorbar(
    forest_df["delta"],
    y,
    xerr=xerr,
    fmt="o",
    capsize=4,
)
ax.axvline(0.0, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(forest_df["label"])
ax.set_xlabel("Paired ΔMAE (model A − model B), weeks")
ax.set_title("All patient-level paired MAE differences with bootstrap 95% CI")
ax.grid(axis="x", alpha=0.25)
for i, row in forest_df.iterrows():
    ax.text(
        row["high"],
        i,
        f"  {row['delta']:.3f} [{row['low']:.3f}, {row['high']:.3f}], "
        f"{significance_text(row['p'])}",
        va="center",
        fontsize=8,
    )
path = save_figure(fig, "07_pairwise_patient_delta_MAE_forest_all_pairs.png")
created_files.append(
    {
        "figure": "07_pairwise_patient_delta_MAE_forest_all_pairs.png",
        "description": "Forest plot of all paired patient MAE differences",
        "path": path,
    }
)

# Focused comparison: each baseline minus CNN. Positive means CNN has lower MAE.
cnn_rows = []
for baseline in [m for m in MODEL_ORDER if m != "CNN 120min"]:
    result = oriented_pair_row(
        pair_patient,
        desired_a=baseline,
        desired_b="CNN 120min",
        delta_col="delta_mae",
        low_col="delta_mae_ci_low",
        high_col="delta_mae_ci_high",
    )
    if result is None:
        continue
    cnn_rows.append({"baseline": baseline, **result})

cnn_df = pd.DataFrame(cnn_rows).sort_values("delta").reset_index(drop=True)
if len(cnn_df):
    fig, ax = plt.subplots(figsize=(10, 6))
    y = np.arange(len(cnn_df))
    xerr = np.vstack(
        [
            cnn_df["delta"].to_numpy() - cnn_df["low"].to_numpy(),
            cnn_df["high"].to_numpy() - cnn_df["delta"].to_numpy(),
        ]
    )
    ax.errorbar(cnn_df["delta"], y, xerr=xerr, fmt="o", capsize=5)
    ax.axvline(0.0, linestyle="--")
    ax.set_yticks(y)
    ax.set_yticklabels(cnn_df["baseline"])
    ax.set_xlabel("Patient ΔMAE: baseline − CNN (weeks)")
    ax.set_title(
        "CNN versus each baseline\nPositive value = CNN has lower patient MAE"
    )
    ax.grid(axis="x", alpha=0.25)
    for i, row in cnn_df.iterrows():
        ax.text(
            row["high"],
            i,
            f"  {row['delta']:.3f} [{row['low']:.3f}, {row['high']:.3f}], "
            f"{significance_text(row['p'])}",
            va="center",
            fontsize=8,
        )
    path = save_figure(fig, "08_CNN_vs_baselines_patient_delta_MAE_forest.png")
    created_files.append(
        {
            "figure": "08_CNN_vs_baselines_patient_delta_MAE_forest.png",
            "description": "Focused patient MAE comparison of CNN against all baselines",
            "path": path,
        }
    )
    cnn_df.to_csv(
        os.path.join(VIS_DIR, "CNN_vs_baselines_patient_delta_MAE_plot_data.csv"),
        index=False,
    )


# ============================================================
# 6. Distribution of patient-level errors
# ============================================================
arrays = [
    patient_error_long.loc[
        patient_error_long["model"] == model, "abs_error"
    ].dropna().to_numpy(dtype=float)
    for model in MODEL_ORDER
]

fig, ax = plt.subplots(figsize=(11, 6))
parts = ax.violinplot(
    arrays,
    positions=np.arange(1, len(MODEL_ORDER) + 1),
    showmeans=False,
    showmedians=True,
    showextrema=True,
)
ax.set_xticks(np.arange(1, len(MODEL_ORDER) + 1))
ax.set_xticklabels(MODEL_ORDER, rotation=30, ha="right")
ax.set_ylabel("Absolute patient error (weeks)")
ax.set_title("Distribution of paired patient-level absolute errors")
ax.grid(axis="y", alpha=0.25)
path = save_figure(fig, "09_patient_absolute_error_violin.png")
created_files.append(
    {
        "figure": "09_patient_absolute_error_violin.png",
        "description": "Violin distributions of patient absolute errors",
        "path": path,
    }
)

# ECDF: lower/left curves represent smaller absolute errors.
fig, ax = plt.subplots(figsize=(10, 6))
for model in MODEL_ORDER:
    values = np.sort(
        patient_error_long.loc[
            patient_error_long["model"] == model, "abs_error"
        ].dropna().to_numpy(dtype=float)
    )
    if len(values) == 0:
        continue
    ecdf = np.arange(1, len(values) + 1) / len(values)
    ax.step(values, ecdf, where="post", label=model)
ax.set_xlabel("Absolute patient error (weeks)")
ax.set_ylabel("Empirical cumulative proportion")
ax.set_title("ECDF of paired patient-level absolute errors")
ax.grid(True, alpha=0.25)
ax.legend()
path = save_figure(fig, "10_patient_absolute_error_ECDF.png")
created_files.append(
    {
        "figure": "10_patient_absolute_error_ECDF.png",
        "description": "ECDF of patient absolute errors",
        "path": path,
    }
)

# How often is each model the best for an individual patient?
wide_errors = wide_patient[["pid"]].copy()
for model in MODEL_ORDER:
    wide_errors[model] = np.abs(
        wide_patient[prediction_column(model)]
        - wide_patient["true_patient_common_CNN"]
    )
wide_errors["best_model"] = wide_errors[MODEL_ORDER].idxmin(axis=1)
best_counts = (
    wide_errors["best_model"]
    .value_counts()
    .reindex(MODEL_ORDER, fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(MODEL_ORDER))
bars = ax.bar(x, best_counts.to_numpy(dtype=int))
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, rotation=30, ha="right")
ax.set_ylabel("Number of patients")
ax.set_title("Model with the smallest absolute error for each patient")
ax.grid(axis="y", alpha=0.25)
for bar, count in zip(bars, best_counts.to_numpy(dtype=int)):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        count,
        str(int(count)),
        ha="center",
        va="bottom",
    )
path = save_figure(fig, "11_best_model_count_per_patient.png")
created_files.append(
    {
        "figure": "11_best_model_count_per_patient.png",
        "description": "Count of patients for whom each model has the smallest error",
        "path": path,
    }
)
wide_errors.to_csv(
    os.path.join(VIS_DIR, "patient_absolute_errors_and_best_model.csv"),
    index=False,
)


# ============================================================
# 7. Patient-level true-versus-predicted plots: one file per model
# ============================================================
for model in MODEL_ORDER:
    pred_col = prediction_column(model)
    x_true, y_pred = finite_xy(
        wide_patient["true_patient_common_CNN"],
        wide_patient[pred_col],
    )
    if len(x_true) == 0:
        continue

    low = float(min(x_true.min(), y_pred.min()))
    high = float(max(x_true.max(), y_pred.max()))

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(x_true, y_pred, alpha=0.65)
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("True patient PMA using common CNN truth (weeks)")
    ax.set_ylabel("Predicted patient PMA (weeks)")
    ax.set_title(f"{model}: patient-level true versus predicted PMA")
    ax.grid(True, alpha=0.25)
    ax.set_xlim(low - 0.25, high + 0.25)
    ax.set_ylim(low - 0.25, high + 0.25)
    filename = f"12_true_vs_pred_{MODEL_TO_SAFE[model]}.png"
    path = save_figure(fig, filename)
    created_files.append(
        {
            "figure": filename,
            "description": f"Patient true-versus-predicted plot for {model}",
            "path": path,
        }
    )


# ============================================================
# 8. Window-level visualization
#
# Native window metrics are visualized in separate compatibility groups.
# A 10-min observation and a 120-min observation are not treated as the same
# statistical unit.
# ============================================================
require_columns(
    summary,
    ["model", "window_mae_native", "window_rmse_native", "window_r2_native"],
    "summary",
)

for group_name, models in WINDOW_GROUPS.items():
    sub = summary[summary["model"].isin(models)].copy()
    sub["model"] = pd.Categorical(sub["model"], models, ordered=True)
    sub = sub.sort_values("model")

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(sub))
    bars = ax.bar(x, sub["window_mae_native"].to_numpy(dtype=float))
    ax.set_xticks(x)
    ax.set_xticklabels(sub["model"].astype(str), rotation=25, ha="right")
    ax.set_ylabel("Native window MAE (weeks)")
    ax.set_title(f"Window-level MAE within compatible group: {group_name}")
    ax.grid(axis="y", alpha=0.25)
    for bar, value in zip(bars, sub["window_mae_native"].to_numpy(dtype=float)):
        if np.isfinite(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value,
                f"{value:.3f}",
                ha="center",
                va="bottom",
            )
    filename = f"20_window_MAE_{group_name}.png"
    path = save_figure(fig, filename)
    created_files.append(
        {
            "figure": filename,
            "description": f"Native window MAE within {group_name}",
            "path": path,
        }
    )

# Patient-clustered window-level paired differences.
complete_window_pairs = pair_window[
    pair_window["status"].isin(["complete_alignment", "partial_alignment"])
].copy()
needed_window_cols = [
    "model_a",
    "model_b",
    "patient_balanced_window_mae_delta_a_minus_b",
    "patient_balanced_window_mae_delta_ci_low",
    "patient_balanced_window_mae_delta_ci_high",
    "wilcoxon_p_value_patient_mean_abs_window_error",
]
if len(complete_window_pairs) and all(
    c in complete_window_pairs.columns for c in needed_window_cols
):
    wforest = complete_window_pairs.copy()
    wforest["label"] = (
        wforest["model_a"].astype(str)
        + " − "
        + wforest["model_b"].astype(str)
    )
    wforest = wforest.sort_values(
        "patient_balanced_window_mae_delta_a_minus_b"
    ).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(11, max(5, 0.65 * len(wforest) + 2)))
    y = np.arange(len(wforest))
    delta = wforest[
        "patient_balanced_window_mae_delta_a_minus_b"
    ].to_numpy(dtype=float)
    low = wforest["patient_balanced_window_mae_delta_ci_low"].to_numpy(dtype=float)
    high = wforest["patient_balanced_window_mae_delta_ci_high"].to_numpy(dtype=float)
    xerr = np.vstack([delta - low, high - delta])
    ax.errorbar(delta, y, xerr=xerr, fmt="o", capsize=5)
    ax.axvline(0.0, linestyle="--")
    ax.set_yticks(y)
    ax.set_yticklabels(wforest["label"])
    ax.set_xlabel(
        "Patient-balanced mean window ΔMAE (model A − model B), weeks"
    )
    ax.set_title(
        "Compatible window-level paired comparisons\n"
        "Bootstrap is clustered at patient level"
    )
    ax.grid(axis="x", alpha=0.25)
    for i, row in wforest.iterrows():
        p = float(row["wilcoxon_p_value_patient_mean_abs_window_error"])
        ax.text(
            float(row["patient_balanced_window_mae_delta_ci_high"]),
            i,
            "  "
            f"{float(row['patient_balanced_window_mae_delta_a_minus_b']):.3f} "
            f"[{float(row['patient_balanced_window_mae_delta_ci_low']):.3f}, "
            f"{float(row['patient_balanced_window_mae_delta_ci_high']):.3f}], "
            f"{significance_text(p)}",
            va="center",
            fontsize=8,
        )
    path = save_figure(fig, "21_pairwise_window_delta_MAE_forest.png")
    created_files.append(
        {
            "figure": "21_pairwise_window_delta_MAE_forest.png",
            "description": "Patient-clustered window-level paired MAE differences",
            "path": path,
        }
    )


# ============================================================
# 9. Optional GA-group heatmap
# ============================================================
if os.path.isfile(PATHS["ga"]):
    ga = pd.read_csv(PATHS["ga"])
    if {"model", "ga_group", "patient_mae"}.issubset(ga.columns):
        ga_order = [
            "EP < 26 wk",
            "VP [26,29) wk",
            "MP [29,32) wk",
            "LP [32,36) wk",
        ]
        matrix = (
            ga.pivot_table(
                index="model",
                columns="ga_group",
                values="patient_mae",
                aggfunc="first",
            )
            .reindex(index=MODEL_ORDER, columns=ga_order)
        )
        if matrix.notna().any().any():
            fig, ax = plt.subplots(figsize=(9, 7))
            im = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto")
            ax.set_xticks(np.arange(len(ga_order)))
            ax.set_xticklabels(ga_order, rotation=25, ha="right")
            ax.set_yticks(np.arange(len(MODEL_ORDER)))
            ax.set_yticklabels(MODEL_ORDER)
            for i in range(len(MODEL_ORDER)):
                for j in range(len(ga_order)):
                    value = matrix.iloc[i, j]
                    if np.isfinite(value):
                        ax.text(j, i, f"{value:.3f}", ha="center", va="center")
            ax.set_title("Patient MAE by birth-GA group")
            fig.colorbar(im, ax=ax, label="Patient MAE (weeks)")
            path = save_figure(fig, "30_GA_group_patient_MAE_heatmap.png")
            created_files.append(
                {
                    "figure": "30_GA_group_patient_MAE_heatmap.png",
                    "description": "GA-group patient MAE heatmap",
                    "path": path,
                }
            )


# ============================================================
# 10. Save manifest and print the most useful figures
# ============================================================
manifest_df = pd.DataFrame(created_files)
manifest_df.to_csv(
    os.path.join(VIS_DIR, "visualization_manifest.csv"),
    index=False,
)

manifest_json = {
    "comparison_output_dir": os.path.abspath(OUTPUT_DIR),
    "visualization_output_dir": os.path.abspath(VIS_DIR),
    "model_order": MODEL_ORDER,
    "main_figures": [
        "01_patient_MAE_common_truth.png",
        "08_CNN_vs_baselines_patient_delta_MAE_forest.png",
        "09_patient_absolute_error_violin.png",
        "06_pairwise_patient_delta_MAE_heatmap.png",
    ],
    "window_policy": (
        "Window-level metrics are visualized separately for 1D 10-min and "
        "2D/CNN 120-min groups. Paired window inference is patient-clustered."
    ),
    "created_files": created_files,
}
with open(
    os.path.join(VIS_DIR, "visualization_manifest.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest_json, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 110)
print("FINAL COMPARISON VISUALIZATIONS COMPLETE")
print("=" * 110)
print("Saved to:", VIS_DIR)
print("\nMain figures to inspect first:")
for filename in manifest_json["main_figures"]:
    path = os.path.join(VIS_DIR, filename)
    print(" -", filename, "[created]" if os.path.isfile(path) else "[not available]")
print("\nAll generated figures:", len(created_files))
print("Manifest:", os.path.join(VIS_DIR, "visualization_manifest.csv"))
print("=" * 110)